In [5]:
import numpy as np
import time

# --- 1. System Parameters (MATCHED to your successful validation) ---
N_RESERVOIR = 1024
# NOTE: LEAK_RATE is 0.3, as used in your successful 
# Butterfly (Cell 5) and Lyapunov (Cell 7) tests.
LEAK_RATE = 0.3          
N_OUTPUT_BITS = 256      # Final output width
TAPPED_NEURONS = np.arange(0, N_RESERVOIR, N_RESERVOIR // N_OUTPUT_BITS) # [0, 4, 8, ...]

# --- STM (Skew Tent Map) Parameters ---
STM_P = 0.4              # Skew parameter for chaos
N_STM_BITS = 32          # Bit-width for the STM stream

# --- 2. Load Model Components ---
print("Loading model and parameters...")
try:
    W_in = np.load("W_in_canonical.npy")
    W_res = np.load("W_res_canonical.npy")
    W_out = np.load("W_out_chua_final.npy") 
except FileNotFoundError:
    print("ERROR: .npy model files not found.")
    print("Please run your ESN_model.ipynb (Cell 2) to generate them first.")
    exit()
    
print("All components loaded.")

def activation_func(x):
    # Use np.tanh as established in your successful training
    return np.tanh(x)

# --- 4. Path A: SRT (LSB Extraction) ---
def extract_raw_bits_srt(r_states_frame):
    """
    Simulates hardware LSB extraction from Q16.16 fixed-point.
    """
    tapped_states = r_states_frame[TAPPED_NEURONS]
    # Emulate Q16.16: Multiply by 2^16, floor, and take LSB (mod 2)
    shifted_int = np.floor(tapped_states * 65536).astype(np.int64) 
    raw_bits = (shifted_int & 1).astype(np.uint8)
    return raw_bits

# --- 5. Path A: ARX_Mixer (Add-Rotate-XOR) ---
def arx_mix(raw_bits_in):
    """
    Simulates the 1-round ARX Mixer on the 256 raw bits.
    """
    packed_bytes = np.packbits(raw_bits_in.astype(bool))
    # Ensure contiguity to prevent a .view() error
    contiguous_bytes = packed_bytes[::-1].copy()
    raw_integer = contiguous_bytes.view(np.uint64)
    a, b, c, d = raw_integer[0], raw_integer[1], raw_integer[2], raw_integer[3]
    
    def ROT(x, n):
        return (x << n) | (x >> (64 - n))

    # --- ARX Quarter-Round (Casting to uint64 prevents overflow errors) ---
    a = (a + b).astype(np.uint64) 
    d = d ^ a; d = ROT(d, 32)
    c = (c + d).astype(np.uint64) 
    b = b ^ c; b = ROT(b, 16)
    a = (a + b).astype(np.uint64) 
    d = d ^ a; d = ROT(d, 8)
    c = (c + d).astype(np.uint64) 
    b = b ^ c; b = ROT(b, 7)

    final_int = np.array([a, b, c, d], dtype=np.uint64)
    mixed_bits = np.unpackbits(final_int.view(np.uint8))[::-1]
    
    return mixed_bits[:N_OUTPUT_BITS].astype(np.uint8)

# --- 6. Path B: Skew Tent Map (STM) ---
def skew_tent_map(x_n, p):
    """Iterates the STM once. Input x_n must be in [0, 1]."""
    if 0 <= x_n < p:
        return x_n / p
    else: # p <= x_n <= 1
        return (1 - x_n) / (1 - p)

def extract_stm_bits(x_stm_float, n_bits):
    """Extracts n_bits LSBs from the STM float state [0, 1]."""
    shift_factor = 2**n_bits
    # Emulate Q0.32: Multiply by 2^32, floor, and get bits
    shifted_int = np.floor(x_stm_float * shift_factor).astype(np.uint64)
    
    # --- FIX ---
    # 1. Convert the 0d (scalar) to a 1d array with one element
    shifted_int_array = np.array([shifted_int], dtype=np.uint64)
    
    # 2. Now .view() will correctly interpret the 8 bytes as an 8-element uint8 array
    byte_array = shifted_int_array.view(np.uint8)

    # 3. Unpack all 64 bits and reverse them (to get LSBs at the start)
    bits_array = np.unpackbits(byte_array)[::-1]
    
    # Return the 32 LSBs
    return bits_array[:n_bits].astype(np.uint8)

# --- 7. Main Simulation Loop and Bitstream Combination ---
def run_hybrid_csprng_simulation(n_steps, n_warmup):
    """Runs the ESN, STM, SRT, and ARX pipeline to generate the final bitstream."""
    print(f"\n--- Starting Full HYBRID CSPRNG Simulation ({n_steps} steps) ---")
    
    # --- Initial Conditions (from successful validation) ---
    u_current = np.array([0.1, 0.2, 1.0]) 
    r_current = np.zeros(N_RESERVOIR)
    x_stm_current = 0.5 # Initial state for the STM [0, 1]
    
    output_stream = []
    start_time = time.time()

    for i in range(n_steps + n_warmup):
        # 1. ESN Core Update
        r_linear = W_res @ r_current + W_in @ u_current
        r_activated = activation_func(r_linear)
        r_current = (1 - LEAK_RATE) * r_current + LEAK_RATE * r_activated
        
        # 2. ESN Readout (Feedback)
        u_current = W_out @ r_current 
        
        # --- Start collecting *after* warmup ---
        if i >= n_warmup:
            # --- Path A: SRT + ARX ---
            raw_bits_A = extract_raw_bits_srt(r_current)
            mixed_stream_A = arx_mix(raw_bits_A) # 256 bits

            # --- Path B: STM ---
            # Scale ESN output (y[0]) to [0, 1] for STM input
            y_x_scaled = (u_current[0] + 15.0) / 30.0 # Assuming ~-15 to 15 range
            y_x_scaled = np.clip(y_x_scaled, 0.0, 1.0) 
            
            # Iterate STM and get 32 raw bits
            x_stm_current = skew_tent_map(y_x_scaled, STM_P)
            raw_stream_B = extract_stm_bits(x_stm_current, N_STM_BITS) # 32 bits

            # --- 3. Final Combination (Feedback XOR Chain) ---
            # Split A into eight 32-bit blocks
            blocks_A = np.split(mixed_stream_A, 8)
            blocks_out = []
            
            # First block is XORed with STM stream
            prev_block = raw_stream_B
            for block in blocks_A:
                new_block = block ^ prev_block
                blocks_out.append(new_block)
                prev_block = new_block
                
            final_keystream = np.concatenate(blocks_out).astype(np.uint8)
            output_stream.append(final_keystream)
        
        if (i + 1) % 1000 == 0:
            print(f"  ... generated step {i+1}")

    final_bitstream = np.concatenate(output_stream).astype(np.uint8)
    end_time = time.time()
    print(f"   Simulation finished in {end_time - start_time:.2f} seconds.")
    return final_bitstream

# --- 8. Execution and NIST Test File Generation ---
# Generate 10.24 million bits (40k steps * 256 bits)
N_STEPS = 5000 
N_WARMUP = 1000
FILE_NAME = "hybrid_csprng_output.bin"

# Run the full simulation
final_bitstream = run_hybrid_csprng_simulation(N_STEPS, N_WARMUP)

# Save the raw bitstream to a binary file for NIST testing
packed_bytes = np.packbits(final_bitstream)
with open(FILE_NAME, "wb") as f:
    f.write(packed_bytes.tobytes())

print(f"\nSuccessfully generated {len(final_bitstream)} bits.")
print(f"File saved as: {FILE_NAME}. Ready for NIST testing!")

Loading model and parameters...
All components loaded.

--- Starting Full HYBRID CSPRNG Simulation (5000 steps) ---
  ... generated step 1000


C:\Users\Dell\AppData\Local\Temp\ipykernel_6816\1937971186.py:59: RuntimeWarning: overflow encountered in scalar add
  a = (a + b).astype(np.uint64)
C:\Users\Dell\AppData\Local\Temp\ipykernel_6816\1937971186.py:65: RuntimeWarning: overflow encountered in scalar add
  c = (c + d).astype(np.uint64)
C:\Users\Dell\AppData\Local\Temp\ipykernel_6816\1937971186.py:61: RuntimeWarning: overflow encountered in scalar add
  c = (c + d).astype(np.uint64)
C:\Users\Dell\AppData\Local\Temp\ipykernel_6816\1937971186.py:63: RuntimeWarning: overflow encountered in scalar add
  a = (a + b).astype(np.uint64)


  ... generated step 2000
  ... generated step 3000
  ... generated step 4000
  ... generated step 5000
  ... generated step 6000
   Simulation finished in 3.59 seconds.

Successfully generated 1280000 bits.
File saved as: hybrid_csprng_output.bin. Ready for NIST testing!


In [1]:
import numpy as np

# --------------------------------------------------
# Configuration
# --------------------------------------------------
Q_SHIFT = 16
Q_SCALE = 1 << Q_SHIFT

H_FILE = "esn_weights.h"
C_FILE = "esn_weights.c"

# --------------------------------------------------
# Load trained weights
# --------------------------------------------------
W_in  = np.load("W_in_canonical.npy")        # (1024, 3)
W_res = np.load("W_res_canonical.npy")       # (1024, 1024)
W_out = np.load("W_out_chua_final.npy")      # (3, 1024)

N_RES, N_IN = W_in.shape
N_OUT = W_out.shape[0]

print("Loaded weights:")
print("W_in :", W_in.shape)
print("W_res:", W_res.shape)
print("W_out:", W_out.shape)

# --------------------------------------------------
# Float → Q16.16
# --------------------------------------------------
def to_q16(x):
    return np.round(x * Q_SCALE).astype(np.int32)

W_in_q  = to_q16(W_in)
W_res_q = to_q16(W_res)
W_out_q = to_q16(W_out)

# --------------------------------------------------
# Write header file
# --------------------------------------------------
with open(H_FILE, "w", encoding="utf-8") as f:
    f.write("#ifndef ESN_WEIGHTS_H\n")
    f.write("#define ESN_WEIGHTS_H\n\n")
    f.write("#include <stdint.h>\n\n")
    f.write(f"#define N_RES {N_RES}\n")
    f.write(f"#define N_IN  {N_IN}\n")
    f.write(f"#define N_OUT {N_OUT}\n\n")
    f.write("extern int32_t W_in[N_RES][N_IN];\n")
    f.write("extern int32_t W_res[N_RES][N_RES];\n")
    f.write("extern int32_t W_out[N_OUT][N_RES];\n\n")
    f.write("#endif\n")

print(f"Wrote {H_FILE}")

# --------------------------------------------------
# Write C source file
# --------------------------------------------------
with open(C_FILE, "w", encoding="utf-8") as f:
    f.write('#include "esn_weights.h"\n\n')

    # W_in
    f.write("int32_t W_in[N_RES][N_IN] = {\n")
    for i in range(N_RES):
        row = ", ".join(str(v) for v in W_in_q[i])
        f.write(f"  {{{row}}},\n")
    f.write("};\n\n")

    # W_res
    f.write("int32_t W_res[N_RES][N_RES] = {\n")
    for i in range(N_RES):
        row = ", ".join(str(v) for v in W_res_q[i])
        f.write(f"  {{{row}}},\n")
    f.write("};\n\n")

    # W_out
    f.write("int32_t W_out[N_OUT][N_RES] = {\n")
    for i in range(N_OUT):
        row = ", ".join(str(v) for v in W_out_q[i])
        f.write(f"  {{{row}}},\n")
    f.write("};\n")

print(f"Wrote {C_FILE}")

print("\nDone. Copy both files into Vitis src/ and build.")


Loaded weights:
W_in : (1024, 3)
W_res: (1024, 1024)
W_out: (3, 1024)
Wrote esn_weights.h
Wrote esn_weights.c

Done. Copy both files into Vitis src/ and build.


In [1]:
import numpy as np

bin_file = "hybrid_csprng_output.bin"
cav_file = "esn_rng.cav"

data = np.fromfile(bin_file, dtype=np.uint8)

bits = np.unpackbits(data)

with open(cav_file, "w") as f:
    for b in bits:
        f.write(str(b))

print("ASCII .cav file written:", cav_file)
print("Total bits:", len(bits))


ASCII .cav file written: esn_rng.cav
Total bits: 1280000
